#### __libraries__

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

In [3]:
from shared_scripts.statistical_analysis import t_margin_of_error, permutation_test_median, iqr_outlier_removal

In [4]:
from shared_scripts.bb2_data_load import BB2_load_into_dataframe

In [5]:
from shared_scripts.additional_functions import extract_significance_symbols

#### __plt params__

In [7]:
plt.rcParams["legend.frameon"] = False
plt.rcParams['legend.fontsize'] = 12
plt.rcParams['axes.labelsize'] = 19
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['xtick.labelsize'] = 16
plt.rcParams['ytick.labelsize'] = 16  
plt.rcParams['font.size'] = 12
plt.rcParams['figure.dpi'] = 1000
plt.rcParams['font.family'] = 'Calibri'

#### __func__

In [9]:
def _2C_data_process(data, **kwargs):
    
    #chck the input data
    if not isinstance(data, pd.DataFrame) or data.empty:
        raise ValueError("input must be a non-empty pandas DataFrame")
        
    if not all(pd.api.types.is_numeric_dtype(dtype) for dtype in data.dtypes):
        raise TypeError("All columns in inout must be numerical")
        
    #assign timepoint
    frq= kwargs.get('frequency', np.NaN)
    indel= kwargs.get('initial_delay', np.NaN)
    if "frequency" not in kwargs or "initial_delay" not in kwargs:
        raise ValueError("frequency and initial_delay must be provided")
    try:
        data= data.assign(timepoint_minutes=data['frame'] * frq + indel).drop(columns=["frame"])
    except Exception as ex:
        raise ValueError("failed to calculate the timepoints in minutes")
        
    #set the index 
    if 'timepoint_minutes' in data.columns:
        data= data.set_index('timepoint_minutes')
    else:
        raise ValueError("dataset is missing a frame column")
        
    #calculate the mean
    try:
        data= data.assign(avg_distance_ctrl= data.iloc[:, 0:3].mean(axis= 1),
                          avg_distance_lata= data.iloc[:, 3:6].mean(axis= 1))
    except Exception as ex:
        print(f'failed to calculate the mean: {ex}')
        data= data.assign(avg_distance_ctrl= np.NaN,
                          avg_distance_lata= np.NaN)
        
    #calculate the MOE95
    try:
        data= data.assign(moe95_ctrl= data.apply(lambda x: t_margin_of_error(x.iloc[0:3]), axis= 1),
                          moe95_lata= data.apply(lambda x: t_margin_of_error(x.iloc[3:6]), axis= 1))
    except Exception as ex:
        print(f'failed to calculate the moe95: {ex}')
        data= data.assign(moe95_ctrl= np.NaN,
                          moe95_lata= np.NaN)
        
    return data

#### __data load__

In [11]:
_6B= BB2_load_into_dataframe('Figure_6B.csv')

In [12]:
_6C= BB2_load_into_dataframe('Figure_6C.csv')

In [13]:
_6D_6F= BB2_load_into_dataframe('Figure_6D_6F.csv')

In [14]:
_6E= BB2_load_into_dataframe('Figure_6E.csv')

#### __data processing__

In [16]:
_6C_final= _2C_data_process(_6C, frequency= 6, initial_delay= 12)

#### __permutation test__

In [18]:
# permutation_test_median(_6B, category_column= 'group',  reference_group= 'dmso_ctrl', tested_parameter= 'alpha', n_resamples= 100000, export= True, path= r"...\Fig_6B_alpha_permutation_test.csv")

In [19]:
# permutation_test_median(_6B, category_column= 'group', reference_group= 'dmso_ctrl', tested_parameter= 'K', n_resamples= 100000, export= True, path= r"...\Fig_6B_K_permutation_test.csv")

#### __significance symbols__

In [21]:
significance_alpha= extract_significance_symbols(path= r"C:\Users\Jakub\Desktop\figure_scripts\permutation_test\Fig_6B_alpha_permutation_test.csv")
significance_k= extract_significance_symbols(path= r"C:\Users\Jakub\Desktop\figure_scripts\permutation_test\Fig_6B_K_permutation_test.csv")

In [49]:
_6D_6F

,group,metal_concentration,timepoint_minutes,percentage_of_cells_with_foci_mean,percentage_of_cells_with_foci_std,percentage_of_cells_with_foci_moe95,percentage_of_cells_with_foci_pvalue,percentage_of_cells_with_foci_significance,avg_number_of_foci_per_cell_mean,avg_number_of_foci_per_cell_std,avg_number_of_foci_per_cell_moe95,avg_number_of_foci_per_cell_pvalue,avg_number_of_foci_per_cell_significance,avg_size_single_focus_mean,avg_size_single_focus_std,avg_size_single_focus_moe95,avg_size_single_focus_pvalue,avg_size_single_focus_significance,Stage,group_index
0,LatA-exposed,0.0 mM,30.0,5.490000,1.114989,2.769786,0.985522,NaN,0.063264,0.014165,0.035188,0.833948,NaN,0.224022,0.027616,0.068602,0.308409,NaN,Formation,3
1,LatA-exposed,0.0 mM,36.0,5.453333,0.745743,1.852529,0.441477,NaN,0.067419,0.010104,0.025101,0.475068,NaN,0.196316,0.023916,0.059410,0.578445,NaN,Formation,3
2,LatA-exposed,0.0 mM,42.0,6.870000,1.801194,4.474414,0.166881,NaN,0.081205,0.020241,0.050281,0.156912,NaN,0.198818,0.011459,0.028467,0.668702,NaN,Formation,3
3,LatA-exposed,0.0 mM,48.0,6.443333,1.140278,2.832607,0.335648,NaN,0.070276,0.012134,0.030142,0.480499,NaN,0.188719,0.024028,0.059690,0.162600,NaN,Formation,3
4,LatA-exposed,0.0 mM,54.0,5.196667,0.522047,1.296837,0.992426,NaN,0.060072,0.008625,0.021427,0.897561,NaN,0.217468,0.018198,0.045205,0.638618,NaN,Formation,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
343,solvent control,0.5 mM,348.0,18.896667,8.508233,21.135623,0.762595,NaN,0.281592,0.126028,0.313071,0.851607,NaN,0.345722,0.039257,0.097520,0.373187,NaN,Clearance,2
344,solvent control,0.5 mM,354.0,17.873333,4.880106,12.122855,0.996352,NaN,0.263068,0.099174,0.246362,0.925897,NaN,0.368663,0.027315,0.067855,0.023338,*,Clearance,2
345,solvent control,0.5 mM,360.0,20.486667,10.011305,24.869461,0.643538,NaN,0.311493,0.194643,0.483521,0.698625,NaN,0.335575,0.049051,0.121849,0.367724,NaN,Clearance,2
346,solvent control,0.5 mM,366.0,18.013333,6.343850,15.758997,0.645278,NaN,0.274044,0.120628,0.299655,0.685744,NaN,0.342973,0.046742,0.116114,0.253198,NaN,Clearance,2


#### __visuals__

In [123]:
def fig6(_6B_data, _6C_data, _6D_6F_data, **kwargs):
    
    #fig + gridspec
    fig= plt.figure(figsize= (12.5, 20), constrained_layout= True)
    gs= gridspec.GridSpec(8, 6, figure=fig)

    ax1= fig.add_subplot(gs[0:2, 0:3])
    ax2= fig.add_subplot(gs[0:2, 3:6])
    ax3= fig.add_subplot(gs[2:4, 0:6])
    ax4= fig.add_subplot(gs[4:6, 0:3])
    ax5= fig.add_subplot(gs[4:6, 3:6])
    ax6= fig.add_subplot(gs[6:7, 0:3])
    ax7= fig.add_subplot(gs[6:7, 3:6])
    ax8= fig.add_subplot(gs[7:8, 0:2])
    ax9= fig.add_subplot(gs[7:8, 2:4])
    ax10= fig.add_subplot(gs[7:8, 4:6])
    
    ###fig 6B
    #alpha
    box_position = 1.5 
    points_bottom, points_top= 0.25, 0.75 
    colors= plt.cm.tab10(np.linspace(0, 1, 10))
    for group in _6B_data.group.unique():
        dark= '#00527C' if group == 'dmso_ctrl' else '#FF781F'
        light= '#DFE9F5' if group == 'dmso_ctrl' else '#FEE7CC'
        data= _6B_data.loc[_6B_data.group== group]
        ax1.boxplot(data.alpha, 
                      positions=[box_position], 
                      widths=0.5,
                      showfliers=False,  
                      patch_artist= True,
                      boxprops=dict(facecolor=light, edgecolor=dark, linewidth=1.5),
                      whiskerprops=dict(color=dark, linewidth=1.5),
                      capprops=dict(color=dark, linewidth=1.5),
                      medianprops=dict(color=dark, linewidth=2))
        symbols_alpha= kwargs.get('significance_symbols_alpha', None)
        if symbols_alpha:
            significance_symbol_position_y= np.max(iqr_outlier_removal(data.alpha))
            ax1.annotate(symbols_alpha.get(group, ''), (box_position, significance_symbol_position_y), fontweight= 'bold', textcoords= 'offset points', xytext= (0, 10), ha= 'center', fontsize= 16)
        x_points = np.random.uniform(points_bottom, points_top, len(data.alpha))
        ax1.scatter(x_points,
                      data.alpha,
                      alpha=0.66,
                      color=light,
                      edgecolors="white",
                      linewidths=0.2,
                      s=12,
                      zorder=3)
        #K
        ax2.boxplot(data.K, 
                      positions=[box_position], 
                      widths=0.5,
                      showfliers=False,  
                      patch_artist= True,
                      boxprops=dict(facecolor=light, edgecolor=dark, linewidth=1.5),
                      whiskerprops=dict(color=dark, linewidth=1.5),
                      capprops=dict(color=dark, linewidth=1.5),
                      medianprops=dict(color=dark, linewidth=2))
        symbols_k= kwargs.get('significance_symbols_k', None)
        if symbols_k:
            significance_symbol_position_y= np.max(iqr_outlier_removal(data.K))
            ax2.annotate(symbols_k.get(group, ''), (box_position, significance_symbol_position_y), fontweight= 'bold', textcoords= 'offset points', xytext= (0, 10), ha= 'center', fontsize= 16)
        ax2.scatter(x_points,
                      data.K,
                      alpha=0.66,
                      color=light,
                      edgecolors="white",
                      linewidths=0.2,
                      s=12,
                      zorder=3)
        box_position += 2
        points_bottom += 2
        points_top += 2
    #rename the groups
    _6B_data= _6B_data.assign(group= np.where(_6B_data.group=='dmso_ctrl', 'As(III)', 'As(III) + LatA'))
    ax1.set_ylabel('alpha', weight= 'bold')
    ax1.set_ylim(bottom=-0.55, top= 2.55)
    ax1.axhline(_6B_data.loc[_6B_data.group=='As(III)', 'alpha'].median(), ls= '--', lw= .66, color="#00527C", alpha= .66, zorder= 5)
    ax1.set_xlim(0, len(_6B_data.group.unique())*2)
    ax1.set_xticks(np.arange(1, len(_6B_data.group.unique())*2, 2))
    ax1.set_xticklabels(_6B_data.group.unique())
    ax1.set_yticks([-0.5, 0.5, 1.5, 2.5])
    ax2.set_ylabel('K', weight= 'bold')
    ax2.set_ylim(bottom=-2, top= 122)
    ax2.axhline(_6B_data.loc[_6B_data.group=='As(III)', 'K'].median(), ls= '--', lw= .66, color="#00527C", alpha= .66, zorder= 5)
    ax2.set_xlim(0, len(_6B_data.group.unique())*2)
    ax2.set_xticks(np.arange(1, len(_6B_data.group.unique())*2, 2))
    ax2.set_xticklabels(_6B_data.group.unique())
    ax2.set_yticks([0, 40, 80, 120])
    
    ###fig 6C
    ax3.errorbar(_6C_data.index, _6C_data.avg_distance_ctrl, lw= 2.5, label= 'As(III)', color= '#00527C')
    y_min_ctrl= _6C_data.avg_distance_ctrl - _6C_data.moe95_ctrl
    y_max_ctrl= _6C_data.avg_distance_ctrl + _6C_data.moe95_ctrl
    ax3.fill_between(_6C_data.index,  y_min_ctrl,  y_max_ctrl,  color='#DFE9F5',  label="margin of error area: As(III)", alpha= .66)
    ax3.errorbar(_6C_data.index, _6C_data.avg_distance_lata, lw= 2.5, label= 'As(III) + LatA', color= '#FF781F')
    y_min_ctrl= _6C_data.avg_distance_lata - _6C_data.moe95_lata
    y_max_ctrl= _6C_data.avg_distance_lata + _6C_data.moe95_lata
    ax3.fill_between(_6C_data.index,  y_min_ctrl,  y_max_ctrl,  color='#FEE7CC',  label="margin of error area: As(III) + LatA", alpha= .66)
    ax3.set_xlabel('time (min)', weight= 'bold')
    ax3.set_ylabel('average movement', weight= 'bold') #pxls/frame
    ax3.set_ylim(0, 8)
    ax3.set_yticks([1, 3, 5, 7])
    ax3.legend(ncol= 2)
    ax3.set_xticks([0,60,120,180,240,300,360])
    
    ###fig 6D a
    solvent_control_data= _6D_6F_data.loc[(_6D_6F.group=='solvent control') & (_6D_6F.metal_concentration=='0.5 mM')]
    cc= round(solvent_control_data.loc[solvent_control_data.Stage=='Formation', 'avg_number_of_foci_per_cell_mean'].corr(solvent_control_data.loc[solvent_control_data.Stage=='Formation', 'avg_size_single_focus_mean'], method= 'pearson', min_periods= 2), 2)
    ax4.scatter(solvent_control_data.loc[solvent_control_data.Stage=='Formation', 'avg_number_of_foci_per_cell_mean'],
                solvent_control_data.loc[solvent_control_data.Stage=='Formation', 'avg_size_single_focus_mean'],
                edgecolor= 'black',
                lw=.15,
                color= '#DFE9F5',
                label= f'FO, {cc}')
    x= solvent_control_data.loc[solvent_control_data.Stage=='Formation', 'avg_number_of_foci_per_cell_mean']
    y= solvent_control_data.loc[solvent_control_data.Stage=='Formation', 'avg_size_single_focus_mean']
    m, b= np.polyfit(x, y, 1)
    ax4.plot(x, 
             m*x+b,
             color= '#00527C',
             linewidth= 2.5)
    
    cc= round(solvent_control_data.loc[solvent_control_data.Stage=='Coalescence/Fusion', 'avg_number_of_foci_per_cell_mean'].corr(solvent_control_data.loc[solvent_control_data.Stage=='Coalescence/Fusion', 'avg_size_single_focus_mean'], method= 'pearson', min_periods= 2), 2)
    ax4.scatter(solvent_control_data.loc[solvent_control_data.Stage=='Coalescence/Fusion', 'avg_number_of_foci_per_cell_mean'],
                solvent_control_data.loc[solvent_control_data.Stage=='Coalescence/Fusion', 'avg_size_single_focus_mean'],
                edgecolor= 'black',
                lw=.15,
                color= '#FEE7CC',
                label= f'C/F, {cc}')
    x= solvent_control_data.loc[solvent_control_data.Stage=='Coalescence/Fusion', 'avg_number_of_foci_per_cell_mean']
    y= solvent_control_data.loc[solvent_control_data.Stage=='Coalescence/Fusion', 'avg_size_single_focus_mean']
    m, b= np.polyfit(x, y, 1)
    ax4.plot(x, 
             m*x+b,
             color= '#FF781F',
             linewidth= 2.5)
    
    cc= round(solvent_control_data.loc[solvent_control_data.Stage=='Clearance', 'avg_number_of_foci_per_cell_mean'].corr(solvent_control_data.loc[solvent_control_data.Stage=='Clearance', 'avg_size_single_focus_mean'], method= 'pearson', min_periods= 2), 2)
    ax4.scatter(solvent_control_data.loc[solvent_control_data.Stage=='Clearance', 'avg_number_of_foci_per_cell_mean'],
                solvent_control_data.loc[solvent_control_data.Stage=='Clearance', 'avg_size_single_focus_mean'],
                edgecolor= 'black',
                lw=.15,
                color= '#D6FED2',
                label= f'CL, {cc}')
    x= solvent_control_data.loc[solvent_control_data.Stage=='Clearance', 'avg_number_of_foci_per_cell_mean']
    y= solvent_control_data.loc[solvent_control_data.Stage=='Clearance', 'avg_size_single_focus_mean']
    m, b= np.polyfit(x, y, 1)
    ax4.plot(x, 
             m*x+b,
             color= '#0C8001',
             linewidth= 2.5)
    solvent_control_control_data= _6D_6F_data.loc[(_6D_6F_data.group=='solvent control') & (_6D_6F_data.metal_concentration=='0.0 mM')]
    ax4.scatter(solvent_control_control_data.loc[:, 'avg_number_of_foci_per_cell_mean'],
                solvent_control_control_data.loc[:, 'avg_size_single_focus_mean'],
                edgecolor= 'black',
                lw=.15,
                color= '#ECECEC',
                label= f'control')
    ax4.set_xlabel('aggregates/cell', weight= 'bold')
    ax4.set_ylabel('aggregate size', weight= 'bold')
    ax4.legend(frameon= True, loc= 'upper right', ncol= 2)
    ax4.set_xlim(0, 3.5)
    ax4.set_ylim(0.12, 0.4)
    ax4.set_yticks([0.2, 0.3, 0.4, 0.5])
    ax4.text(00.70, 0.1, 
         f'As(III)',
         transform=ax4.transAxes,
         fontweight= 'bold', fontsize= 18,
         verticalalignment='center',
         horizontalalignment='center')
    ax4.set_xlim(0, 3)
    
    ###fig 6D b
    lata_exposed_data= _6D_6F_data.loc[(_6D_6F.group=='LatA-exposed') & (_6D_6F.metal_concentration=='0.5 mM')]
    cc= round(lata_exposed_data.loc[lata_exposed_data.Stage=='Formation', 'avg_number_of_foci_per_cell_mean'].corr(lata_exposed_data.loc[lata_exposed_data.Stage=='Formation', 'avg_size_single_focus_mean'], method= 'pearson', min_periods= 2), 2)
    ax5.scatter(lata_exposed_data.loc[lata_exposed_data.Stage=='Formation', 'avg_number_of_foci_per_cell_mean'],
                lata_exposed_data.loc[lata_exposed_data.Stage=='Formation', 'avg_size_single_focus_mean'],
                edgecolor= 'black',
                lw=.15,
                color= '#DFE9F5',
                label= f'FO, {cc}')
    x= lata_exposed_data.loc[lata_exposed_data.Stage=='Formation', 'avg_number_of_foci_per_cell_mean']
    y= lata_exposed_data.loc[lata_exposed_data.Stage=='Formation', 'avg_size_single_focus_mean']
    m, b= np.polyfit(x, y, 1)
    ax5.plot(x, 
             m*x+b,
             color= '#00527C',
             linewidth= 2.5)
    
    cc= round(lata_exposed_data.loc[lata_exposed_data.Stage=='Coalescence/Fusion', 'avg_number_of_foci_per_cell_mean'].corr(lata_exposed_data.loc[lata_exposed_data.Stage=='Coalescence/Fusion', 'avg_size_single_focus_mean'], method= 'pearson', min_periods= 2), 2)
    ax5.scatter(lata_exposed_data.loc[lata_exposed_data.Stage=='Coalescence/Fusion', 'avg_number_of_foci_per_cell_mean'],
                lata_exposed_data.loc[lata_exposed_data.Stage=='Coalescence/Fusion', 'avg_size_single_focus_mean'],
                edgecolor= 'black',
                lw=.15,
                color= '#FEE7CC',
                label= f'C/F, {cc}')
    x= lata_exposed_data.loc[lata_exposed_data.Stage=='Coalescence/Fusion', 'avg_number_of_foci_per_cell_mean']
    y= lata_exposed_data.loc[lata_exposed_data.Stage=='Coalescence/Fusion', 'avg_size_single_focus_mean']
    m, b= np.polyfit(x, y, 1)
    ax5.plot(x, 
             m*x+b,
             color= '#FF781F',
             linewidth= 2.5)
    
    cc= round(lata_exposed_data.loc[lata_exposed_data.Stage=='Clearance', 'avg_number_of_foci_per_cell_mean'].corr(lata_exposed_data.loc[lata_exposed_data.Stage=='Clearance', 'avg_size_single_focus_mean'], method= 'pearson', min_periods= 2), 2)
    ax5.scatter(lata_exposed_data.loc[lata_exposed_data.Stage=='Clearance', 'avg_number_of_foci_per_cell_mean'],
                lata_exposed_data.loc[lata_exposed_data.Stage=='Clearance', 'avg_size_single_focus_mean'],
                edgecolor= 'black',
                lw=.15,
                color= '#D6FED2',
                label= f'CL, {cc}')
    x= lata_exposed_data.loc[lata_exposed_data.Stage=='Clearance', 'avg_number_of_foci_per_cell_mean']
    y= lata_exposed_data.loc[lata_exposed_data.Stage=='Clearance', 'avg_size_single_focus_mean']
    m, b= np.polyfit(x, y, 1)
    ax5.plot(x, 
             m*x+b,
             color= '#0C8001',
             linewidth= 2.5)
    lata_exposed_control_data= _6D_6F_data.loc[(_6D_6F_data.group=='LatA-exposed') & (_6D_6F_data.metal_concentration=='0.0 mM')]
    ax5.scatter(lata_exposed_control_data.loc[:, 'avg_number_of_foci_per_cell_mean'],
                lata_exposed_control_data.loc[:, 'avg_size_single_focus_mean'],
                edgecolor= 'black',
                lw=.15,
                color= '#ECECEC',
                label= f'control')
    ax5.set_xlabel('aggregates/cell', weight= 'bold')
    ax5.set_ylabel('aggregate size', weight= 'bold')
    ax5.legend(frameon= True, loc= 'upper right', ncol= 2)
    ax5.set_xlim(0, 3.5)
    ax5.set_ylim(0.12, 0.4)
    ax5.set_yticks([0.2, 0.3, 0.4, 0.5])
    ax5.text(00.70, 0.1, 
         f'As(III) + LatA',
         transform=ax5.transAxes,
         fontweight= 'bold', fontsize= 18,
         verticalalignment='center',
         horizontalalignment='center')
    ax5.set_xlim(0, 3)

In [125]:
fig6(_6B, 
     _6C_final,
     _6D_6F,
     significance_symbols_alpha= significance_alpha,
     significance_symbols_k= significance_k)